# ChessGPT â€” Board-State Aware

Architecture upgrade: 64 board-state tokens interleaved before every move.  
Model sees exact piece positions -> learns legality from data, no explicit penalty.

**What changed vs previous notebooks**
- Vocab: 4546 moves + 13 piece tokens = **4559**
- Sequence: `[GS, board0x64, move1, board1x64, move2, ...]` â€” model reads the board
- Context: **512** tokens (~7 full board+move pairs)
- Model: **384 emb / 8 heads / 8 layers -> ~14M params**
- Loss: CE on **move tokens only** (board tokens = context, not targets)
- Training: **AMP mixed precision** + grad clip + cosine LR
- Eval: **ACPL via Stockfish** (centipawn loss, not just cross-entropy)

**Runtime -> T4 GPU before running.**

In [ ]:
!pip install chess matplotlib tqdm zstandard -q

## Source files

In [ ]:
import os
os.makedirs('src', exist_ok=True)
os.makedirs('data', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
open('src/__init__.py', 'w').close()

with open('src/vocab.py',    'w') as f: f.write('\nimport os, chess\n\nGAME_START     = "<|game_start|>"\nGAME_END       = "<|game_end|>"\nSPECIAL_TOKENS = [GAME_START, GAME_END]\nPIECE_SYMBOLS  = [\'.\', \'P\', \'N\', \'B\', \'R\', \'Q\', \'K\', \'p\', \'n\', \'b\', \'r\', \'q\', \'k\']\n\ndef _enumerate_all_moves():\n    files, ranks, promos = "abcdefgh", "12345678", "qrbn"\n    moves = []\n    for f1 in files:\n        for r1 in ranks:\n            for f2 in files:\n                for r2 in ranks:\n                    if f1+r1 != f2+r2:\n                        moves.append(f1+r1+f2+r2)\n    for f1 in files:\n        for f2 in files:\n            for p in promos:\n                moves.append(f1+"7"+f2+"8"+p)\n                moves.append(f1+"2"+f2+"1"+p)\n    return moves\n\nclass ChessTokenizer:\n    def __init__(self):\n        move_vocab       = SPECIAL_TOKENS + sorted(set(_enumerate_all_moves()))\n        self.move_to_id  = {m: i for i, m in enumerate(move_vocab)}\n        self.piece_to_id = {p: len(move_vocab)+i for i,p in enumerate(PIECE_SYMBOLS)}\n        all_vocab        = move_vocab + PIECE_SYMBOLS\n        self.id_to_token = {i: t for i,t in enumerate(all_vocab)}\n        self.vocab_size  = len(all_vocab)   # 4559\n        self.n_moves     = len(move_vocab)  # 4546\n        self.game_start_id = self.move_to_id[GAME_START]\n        self.game_end_id   = self.move_to_id[GAME_END]\n\n    def board_to_ids(self, board):\n        return [self.piece_to_id[board.piece_at(sq).symbol()\n                if board.piece_at(sq) else \'.\']\n                for sq in chess.SQUARES]\n\n    def is_move_id(self, tid):\n        return 2 <= tid < self.n_moves\n')
with open('src/dataset.py',  'w') as f: f.write('\nimport chess, chess.pgn, numpy as np, torch\nfrom torch.utils.data import Dataset, DataLoader\nfrom src.vocab import ChessTokenizer\n\ndef _elo(v):\n    try: return int(v or 0)\n    except: return 0\n\ndef parse_pgn(path, max_games=None, min_elo=0):\n    from tqdm.auto import tqdm\n    games, skipped = [], 0\n    with open(path, encoding="utf-8", errors="ignore") as f:\n        pbar = tqdm(unit="games", desc="Parsing PGN")\n        while True:\n            game = chess.pgn.read_game(f)\n            if game is None: break\n            if min_elo > 0:\n                w = _elo(game.headers.get("WhiteElo", 0))\n                b = _elo(game.headers.get("BlackElo", 0))\n                if min(w, b) < min_elo:\n                    skipped += 1; pbar.update()\n                    pbar.set_postfix(kept=len(games), skipped=skipped); continue\n            moves = [m.uci() for m in game.mainline_moves()]\n            if moves: games.append(moves)\n            pbar.update(); pbar.set_postfix(kept=len(games), skipped=skipped)\n            if max_games and len(games) >= max_games: break\n        pbar.close()\n    print(f"Loaded {len(games)} games (skipped {skipped} below ELO {min_elo})")\n    return games\n\ndef games_to_stream(games, tokenizer):\n    # Stream per game: [GS, board0x64, m1, board1x64, m2, ..., GE]\n    # is_move[i] = True only at move-token positions\n    token_ids, is_move = [], []\n    for moves in games:\n        board = chess.Board()\n        token_ids.append(tokenizer.game_start_id); is_move.append(False)\n        token_ids.extend(tokenizer.board_to_ids(board)); is_move.extend([False]*64)\n        for uci in moves:\n            if uci not in tokenizer.move_to_id: continue\n            token_ids.append(tokenizer.move_to_id[uci]); is_move.append(True)\n            board.push(chess.Move.from_uci(uci))\n            token_ids.extend(tokenizer.board_to_ids(board)); is_move.extend([False]*64)\n        token_ids.append(tokenizer.game_end_id); is_move.append(False)\n    return np.array(token_ids, dtype=np.int16), np.array(is_move, dtype=bool)\n\nclass ChessDataset(Dataset):\n    # Stores full arrays; slices in __getitem__ â€” O(1) RAM per window\n    def __init__(self, toks, is_move, start, end, ctx, stride):\n        self.toks, self.is_move = toks, is_move\n        self.ctx    = ctx\n        self.starts = list(range(start, end - ctx, stride))\n\n    def __len__(self): return len(self.starts)\n\n    def __getitem__(self, idx):\n        i = self.starts[idx]\n        x = torch.tensor(self.toks[i:i+self.ctx].astype(np.int64))\n        y = torch.tensor(self.toks[i+1:i+self.ctx+1].astype(np.int64))\n        m = torch.tensor(self.is_move[i+1:i+self.ctx+1])\n        return x, y, m\n\ndef create_dataloader(toks, is_move, start, end, ctx=512, stride=256,\n                      batch_size=32, shuffle=True, drop_last=True, num_workers=2):\n    ds = ChessDataset(toks, is_move, start, end, ctx, stride)\n    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,\n                      drop_last=drop_last, num_workers=num_workers, pin_memory=True)\n')
with open('src/model.py',    'w') as f: f.write('\nimport torch, torch.nn as nn\n\nclass MultiHeadAttention(nn.Module):\n    def __init__(self, d_in, d_out, ctx, dropout, heads, qkv_bias=False):\n        super().__init__()\n        assert d_out % heads == 0\n        self.d_out, self.heads, self.head_dim = d_out, heads, d_out//heads\n        self.W_q = nn.Linear(d_in, d_out, bias=qkv_bias)\n        self.W_k = nn.Linear(d_in, d_out, bias=qkv_bias)\n        self.W_v = nn.Linear(d_in, d_out, bias=qkv_bias)\n        self.proj = nn.Linear(d_out, d_out)\n        self.drop = nn.Dropout(dropout)\n        self.register_buffer("mask", torch.triu(torch.ones(ctx, ctx), diagonal=1))\n    def forward(self, x):\n        b, n, _ = x.shape\n        k = self.W_k(x).view(b,n,self.heads,self.head_dim).transpose(1,2)\n        q = self.W_q(x).view(b,n,self.heads,self.head_dim).transpose(1,2)\n        v = self.W_v(x).view(b,n,self.heads,self.head_dim).transpose(1,2)\n        s = q @ k.transpose(2,3)\n        s.masked_fill_(self.mask.bool()[:n,:n], -torch.inf)\n        w = self.drop(torch.softmax(s / k.shape[-1]**0.5, dim=-1))\n        return self.proj((w@v).transpose(1,2).contiguous().view(b,n,self.d_out))\n\nclass LayerNorm(nn.Module):\n    def __init__(self, d):\n        super().__init__()\n        self.scale = nn.Parameter(torch.ones(d))\n        self.shift = nn.Parameter(torch.zeros(d))\n    def forward(self, x):\n        m, v = x.mean(-1,keepdim=True), x.var(-1,keepdim=True,unbiased=False)\n        return self.scale*(x-m)/torch.sqrt(v+1e-5)+self.shift\n\nclass GELU(nn.Module):\n    def forward(self, x):\n        return 0.5*x*(1+torch.tanh((2/torch.pi)**0.5*(x+0.044715*x**3)))\n\nclass FeedForward(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]), GELU(),\n            nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"]))\n    def forward(self, x): return self.net(x)\n\nclass TransformerBlock(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.att  = MultiHeadAttention(cfg["emb_dim"],cfg["emb_dim"],\n                                        cfg["context_length"],cfg["drop_rate"],\n                                        cfg["n_heads"],cfg["qkv_bias"])\n        self.ff   = FeedForward(cfg)\n        self.n1   = LayerNorm(cfg["emb_dim"])\n        self.n2   = LayerNorm(cfg["emb_dim"])\n        self.drop = nn.Dropout(cfg["drop_rate"])\n    def forward(self, x):\n        x = x + self.drop(self.att(self.n1(x)))\n        x = x + self.drop(self.ff(self.n2(x)))\n        return x\n\nclass GPTModel(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.tok_emb  = nn.Embedding(cfg["vocab_size"],    cfg["emb_dim"])\n        self.pos_emb  = nn.Embedding(cfg["context_length"],cfg["emb_dim"])\n        self.drop     = nn.Dropout(cfg["drop_rate"])\n        self.blocks   = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])\n        self.norm     = LayerNorm(cfg["emb_dim"])\n        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)\n    def forward(self, idx):\n        b, t = idx.shape\n        x = self.tok_emb(idx) + self.pos_emb(torch.arange(t, device=idx.device))\n        return self.out_head(self.norm(self.blocks(self.drop(x))))\n')
with open('src/generate.py', 'w') as f: f.write('\nimport torch, chess\nfrom src.vocab import GAME_START\n\ndef _legal_ids(board, tokenizer):\n    return [tokenizer.move_to_id[m.uci()] for m in board.legal_moves\n            if m.uci() in tokenizer.move_to_id]\n\ndef generate_move(model, board, ctx_ids, tokenizer, device, temperature=1.0):\n    model.eval()\n    ctx = model.pos_emb.weight.shape[0]\n    idx = torch.tensor(ctx_ids[-ctx:], dtype=torch.long, device=device).unsqueeze(0)\n    with torch.no_grad():\n        logits = model(idx)[0, -1, :]\n    legal = _legal_ids(board, tokenizer)\n    if not legal: raise ValueError("No legal moves")\n    mask = torch.full_like(logits, float("-inf"))\n    mask[legal] = logits[legal]\n    return tokenizer.id_to_move[torch.multinomial(torch.softmax(mask/temperature,-1),1).item()]\n\ndef generate_sample_game(model, tokenizer, device, num_moves=20, temperature=1.0):\n    board   = chess.Board()\n    ctx_ids = [tokenizer.game_start_id] + tokenizer.board_to_ids(board)\n    moves   = []\n    for _ in range(num_moves):\n        if board.is_game_over(): break\n        try:\n            uci = generate_move(model, board, ctx_ids, tokenizer, device, temperature)\n        except: break\n        board.push(chess.Move.from_uci(uci))\n        ctx_ids.append(tokenizer.move_to_id[uci])\n        ctx_ids.extend(tokenizer.board_to_ids(board))\n        moves.append(uci)\n    return moves\n')
print('Source files written.')


## Config

In [ ]:
import torch
from src.vocab import ChessTokenizer

tokenizer = ChessTokenizer()
print(f'Vocab size: {tokenizer.vocab_size}')  # 4559

CHESS_GPT_CONFIG = {
    'vocab_size':     tokenizer.vocab_size,
    'context_length': 512,
    'emb_dim':        384,
    'n_heads':        8,
    'n_layers':       8,
    'drop_rate':      0.1,
    'qkv_bias':       False,
}

TRAIN_SETTINGS = {
    'learning_rate': 3e-4,
    'weight_decay':  0.1,
    'num_epochs':    10,
    'batch_size':    32,
    'eval_freq':     500,
    'eval_iter':     20,
    'min_elo':       2000,
    'max_games':     100000,
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


## Download Lichess 2026-04

In [ ]:
import os, zstandard, requests

PGN_PATH = 'data/lichess_2026-04.pgn'
URL      = 'https://database.lichess.org/standard/lichess_db_standard_rated_2026-04.pgn.zst'

def download_lichess(url, pgn_path):
    zst = pgn_path + '.zst'
    headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36'}
    r = requests.get(url, headers=headers, stream=True, timeout=600)
    print(f'HTTP {r.status_code}')
    if r.status_code != 200: raise RuntimeError(f'HTTP {r.status_code}')
    total, received = int(r.headers.get('Content-Length', 0)), 0
    with open(zst, 'wb') as f:
        for chunk in r.iter_content(1<<20):
            f.write(chunk); received += len(chunk)
            if total: print(f'  {received/1e6:.0f}/{total/1e6:.0f} MB', end='\r')
    print('\nDecompressing...')
    with open(zst,'rb') as s, open(pgn_path,'wb') as d:
        zstandard.ZstdDecompressor().copy_stream(s, d)
    os.remove(zst)
    print(f'Done: {os.path.getsize(pgn_path)/1e6:.0f} MB')

if not os.path.exists(PGN_PATH):
    download_lichess(URL, PGN_PATH)
else:
    print(f'Already exists: {os.path.getsize(PGN_PATH)/1e6:.0f} MB')


## Load + tokenize (board-state interleaved)

Sequence is ~65x longer than pure move tokens. 100k games at 2000+ ELO.

In [ ]:
from src.dataset import parse_pgn, games_to_stream, create_dataloader

games = parse_pgn(
    PGN_PATH,
    max_games=TRAIN_SETTINGS['max_games'],
    min_elo=TRAIN_SETTINGS['min_elo'],
)

toks, is_move = games_to_stream(games, tokenizer)
del games
print(f'Total tokens: {len(toks):,}  |  Move tokens: {is_move.sum():,}')
print(f'RAM: toks={toks.nbytes/1e6:.0f} MB  is_move={is_move.nbytes/1e6:.0f} MB')

split   = int(0.9 * len(toks))
ctx, bs = CHESS_GPT_CONFIG['context_length'], TRAIN_SETTINGS['batch_size']

train_loader = create_dataloader(toks, is_move, 0, split, ctx, ctx//2, bs,
                                  shuffle=True,  drop_last=True)
val_loader   = create_dataloader(toks, is_move, split, len(toks), ctx, ctx, bs,
                                  shuffle=False, drop_last=False)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')


## Build model

In [ ]:
from src.model import GPTModel
import torch

torch.manual_seed(42)
model = GPTModel(CHESS_GPT_CONFIG).to(device)
params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {params:,}')  # ~14M


## Train (AMP + grad clip + cosine LR)

Loss computed on move positions only. Board tokens inform predictions, not penalise them.

In [ ]:
import torch.nn as nn
from src.generate import generate_sample_game

use_amp = device.type == 'cuda'
scaler  = torch.amp.GradScaler('cuda', enabled=use_amp)

def calc_loss_batch(x, y, move_mask, model, device):
    # CE loss on move-token positions only; board tokens are context not targets
    with torch.amp.autocast('cuda', enabled=use_amp):
        logits = model(x.to(device))
    y, mask = y.to(device), move_mask.to(device)
    if not mask.any():
        return torch.tensor(0.0, device=device, requires_grad=True)
    return nn.functional.cross_entropy(logits[mask], y[mask])

def calc_loss_loader(loader, model, device, n=None):
    total, cnt = 0.0, 0
    nb = min(n or len(loader), len(loader))
    for i, (x, y, m) in enumerate(loader):
        if i >= nb: break
        total += calc_loss_batch(x, y, m, model, device).item(); cnt += 1
    return total/cnt if cnt else float('nan')

def train(model, train_loader, val_loader, optimizer, scheduler, device,
          num_epochs, eval_freq, eval_iter, tokenizer):
    train_losses, val_losses, tokens_seen_list = [], [], []
    tokens_seen, step = 0, -1
    for epoch in range(num_epochs):
        model.train()
        for x, y, m in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(x, y, m, model, device)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            tokens_seen += m.sum().item(); step += 1
            if step % eval_freq == 0:
                model.eval()
                with torch.no_grad():
                    tl = calc_loss_loader(train_loader, model, device, eval_iter)
                    vl = calc_loss_loader(val_loader,   model, device, eval_iter)
                model.train()
                train_losses.append(tl); val_losses.append(vl)
                tokens_seen_list.append(tokens_seen)
                lr = optimizer.param_groups[0]['lr']
                print(f'Ep {epoch+1} (Step {step:06d}): '
                      f'Train {tl:.3f}  Val {vl:.3f}  LR {lr:.2e}')
        scheduler.step()
        sample = generate_sample_game(model, tokenizer, device, num_moves=10)
        print(f'  Sample: {" ".join(sample)}')
        torch.save(model.state_dict(), f'checkpoints/chess_gpt_bs_ep{epoch+1}.pth')
        print(f'  Checkpoint saved.')
    return train_losses, val_losses, tokens_seen_list

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=TRAIN_SETTINGS['learning_rate'],
    weight_decay=TRAIN_SETTINGS['weight_decay'],
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=TRAIN_SETTINGS['num_epochs'])

train_losses, val_losses, tokens_seen = train(
    model, train_loader, val_loader, optimizer, scheduler, device,
    num_epochs=TRAIN_SETTINGS['num_epochs'],
    eval_freq=TRAIN_SETTINGS['eval_freq'],
    eval_iter=TRAIN_SETTINGS['eval_iter'],
    tokenizer=tokenizer,
)


## Loss plot

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9,4))
steps = list(range(len(train_losses)))
ax.plot(steps, train_losses, label='Train')
ax.plot(steps, val_losses, linestyle='-.', label='Val')
ax.set_xlabel('Eval step'); ax.set_ylabel('Loss (move tokens only)')
ax.set_title('Queen's GAMBIT Board-State 2000+ ELO'); ax.legend()
plt.tight_layout(); plt.savefig('loss.png', dpi=150); plt.show()
print(f'Final  train: {train_losses[-1]:.3f}  val: {val_losses[-1]:.3f}')


## Watch model play

In [ ]:
import chess
from src.generate import generate_move

model.eval()
board    = chess.Board()
ctx_ids  = [tokenizer.game_start_id] + tokenizer.board_to_ids(board)
move_num = 1

for _ in range(400):
    if board.is_game_over(): break
    is_white = board.turn == chess.WHITE
    uci  = generate_move(model, board, ctx_ids, tokenizer, device, temperature=0.8)
    move = chess.Move.from_uci(uci)
    san  = board.san(move)
    board.push(move)
    ctx_ids.append(tokenizer.move_to_id[uci])
    ctx_ids.extend(tokenizer.board_to_ids(board))
    if is_white: print(f'{move_num:3}. {san}', end='')
    else:        print(f'  {san}'); move_num += 1

print(f'\nResult: {board.result()}')
if board.is_checkmate():     print('CHECKMATE')
elif board.is_stalemate():   print('STALEMATE')
elif board.is_repetition():  print('DRAW - repetition')
elif board.is_fifty_moves(): print('DRAW - 50-move rule')


## ACPL evaluation (Stockfish)

Measures **Average Centipawn Loss**: how many centipawns worse than Stockfish's best move the model plays, on average.  
**Lower = stronger.** Typical benchmarks:
- Random legal moves: ~300
- Beginner: ~200
- Club player: ~100
- 1800 ELO: ~50
- 2500 ELO: ~20

Run this cell after training to compare across ELO filter experiments.

In [ ]:
!apt-get install -y stockfish -q
!pip install stockfish -q

In [ ]:
from stockfish import Stockfish
import chess
from src.generate import generate_move

def eval_acpl(model, tokenizer, device, n_games=10, sf_depth=8, temperature=0.5):
    # Average Centipawn Loss: lower = stronger play
    # CPL per move = max(0, eval(SF best) - eval(model move)) from mover's POV
    sf = Stockfish(depth=sf_depth, parameters={'Threads': 2, 'Hash': 64})
    total_cpl, n_moves = 0, 0
    model.eval()
    for g in range(n_games):
        board   = chess.Board()
        ctx_ids = [tokenizer.game_start_id] + tokenizer.board_to_ids(board)
        for _ in range(80):
            if board.is_game_over(): break
            fen = board.fen()
            sf.set_fen_position(fen)
            best_uci = sf.get_best_move()
            if not best_uci: break
            try:
                model_uci = generate_move(model, board, ctx_ids, tokenizer, device, temperature)
            except: break
            def cp_for_mover(move_uci):
                # Returns centipawns for the side that just moved (higher = better for them)
                b = board.copy()
                b.push(chess.Move.from_uci(move_uci))
                sf.set_fen_position(b.fen())
                ev = sf.get_evaluation()
                if ev['type'] == 'cp':
                    return -ev['value']  # negate: opponent's turn, flip perspective
                return 10000 if ev['value'] < 0 else -10000
            e_best  = cp_for_mover(best_uci)
            e_model = cp_for_mover(model_uci)
            total_cpl += max(0, e_best - e_model)
            n_moves   += 1
            board.push(chess.Move.from_uci(model_uci))
            ctx_ids.append(tokenizer.move_to_id[model_uci])
            ctx_ids.extend(tokenizer.board_to_ids(board))
        print(f'Game {g+1}/{n_games}  running ACPL: {total_cpl/max(n_moves,1):.1f}')
    acpl = total_cpl / n_moves if n_moves else float('inf')
    print(f'\nFinal ACPL: {acpl:.1f} cp  ({n_moves} moves across {n_games} games)')
    print('Reference: beginner ~300, club ~100, 1800 ELO ~50, 2500 ELO ~20')
    return acpl

acpl = eval_acpl(model, tokenizer, device, n_games=10, sf_depth=8, temperature=0.5)


## Save to Google Drive (optional)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp checkpoints/chess_gpt_bs_ep10.pth /content/drive/MyDrive/
